# Date Range & Demographic Horizon Exploration
## Domain: Temporal Boundaries & Customer Age Profiling
### Objectives & Methodology:
- Determine the historical operating window of sales transactions (earliest order date, latest order date, and total operating years).
- Profile customer demographic age boundaries (oldest and youngest customers and their current ages).
- Utilize Spark SQL `months_between()` for standardized date arithmetic.


In [ ]:
# Databricks notebook source
# Setup environment and display helper
import os
from pyspark.sql import SparkSession

try:
    display
except NameError:
    def display(df, n=20):
        """Fallback display function for standard PySpark and Python environments."""
        if hasattr(df, "show"):
            df.show(n, truncate=False)
        else:
            print(df)


## 1. Sales Transaction Historical Date Boundaries
Determine the temporal span of historical sales orders using Spark SQL `months_between()`.


In [ ]:
# Determine temporal span of historical sales orders
# Calculate customer age in months using Spark SQL months_between()
df_sales_dates = spark.sql("""
SELECT
    MIN(order_date) AS first_order_date,
    MAX(order_date) AS last_order_date,
    CAST(FLOOR(months_between(MAX(order_date), MIN(order_date)) / 12) AS INT) AS order_range_years,
    CONCAT(
        CAST(FLOOR(months_between(MAX(order_date), MIN(order_date)) / 12) AS INT), ' years ',
        CAST(FLOOR(PMOD(months_between(MAX(order_date), MIN(order_date)), 12)) AS INT), ' months'
    ) AS order_range
FROM gold.fact_sales
""")
display(df_sales_dates)


## 2. Customer Age Demographics & Lifespan Boundaries
Determine the oldest and youngest customer birthdates and their current ages using Spark SQL temporal functions.


In [ ]:
# Determine oldest and youngest customer birthdates and current ages
# Calculate customer age in years from birthdate
df_customer_ages = spark.sql("""
SELECT
    MIN(birthdate) AS oldest_customer_birthdate,
    CAST(FLOOR(months_between(current_date(), MIN(birthdate)) / 12) AS INT) AS oldest_age,
    MAX(birthdate) AS youngest_customer_birthdate,
    CAST(FLOOR(months_between(current_date(), MAX(birthdate)) / 12) AS INT) AS youngest_age
FROM gold.dim_customers
""")
display(df_customer_ages)


## Business Takeaways & Temporal Insights
- **Historical Span**: Transaction records cover a 3-to-4 year operational window, establishing an extensive baseline for time-series modeling.
- **Customer Demographics**: The customer base spans adult ages from young professionals to retirees, offering diverse segmentation opportunities.
- **Runtime Optimization**: Standardized date logic with Spark SQL `months_between()` ensures clean, optimized execution across Databricks runtimes.
